# Chapter 18 — Hyperparameter Tuning
**Track:** ML from Scratch · California Housing dataset

Every dial in a neural network, what it does, and where to start. Mirrors `README.md`.


## Core Idea

Parameters ($W, b$) are **learned**. Hyperparameters are **chosen** before training starts and control *how* learning happens.

Tune in this order — cheapest, highest-leverage first:

```
learning rate → batch size → optimiser → initialiser
    → architecture (layers, units, layer type)
    → regularisation (dropout, weight decay, early stopping)
    → loss choice → more data
```


In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore", category=UserWarning)  # sklearn MLP convergence warnings
np.random.seed(42)

housing = fetch_california_housing()
X, y = housing.data, housing.target
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler().fit(X_tr_raw)
X_tr = scaler.transform(X_tr_raw)
X_te = scaler.transform(X_te_raw)

print(f"Train: {X_tr.shape}   Test: {X_te.shape}   target range: [{y.min():.2f}, {y.max():.2f}]")


## 1 · Learning rate sweep

The single most impactful knob. Too small → loss crawls. Too large → diverges / oscillates.


In [ ]:
lrs = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
lr_results = []

fig, ax = plt.subplots(figsize=(9, 5))
for lr in lrs:
    m = MLPRegressor(hidden_layer_sizes=(64, 32), solver="adam",
                     learning_rate_init=lr, max_iter=60, random_state=42)
    m.fit(X_tr, y_tr)
    r2 = r2_score(y_te, m.predict(X_te))
    ax.plot(m.loss_curve_, label=f"η = {lr:g}")
    lr_results.append((lr, m.loss_curve_[-1], r2))

ax.set_xlabel("epoch"); ax.set_ylabel("training loss (MSE)")
ax.set_yscale("log"); ax.set_title("Learning rate sweep — Adam")
ax.legend(); ax.grid(True, alpha=0.3); plt.show()

pd.DataFrame(lr_results, columns=["learning_rate", "final_train_loss", "test_R2"])


## 2 · Optimiser comparison

SGD → SGD+Momentum → Adam. Each optimiser gets its own well-tuned learning rate (SGD needs ~10× higher than Adam).


In [ ]:
opt_configs = [
    ("SGD",            dict(solver="sgd",  learning_rate_init=1e-2, momentum=0.0)),
    ("SGD + Momentum", dict(solver="sgd",  learning_rate_init=1e-2, momentum=0.9)),
    ("Adam",           dict(solver="adam", learning_rate_init=1e-3)),
]

fig, ax = plt.subplots(figsize=(9, 5))
opt_results = []
for name, kw in opt_configs:
    m = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=80, random_state=42, **kw)
    m.fit(X_tr, y_tr)
    r2 = r2_score(y_te, m.predict(X_te))
    ax.plot(m.loss_curve_, label=f"{name}  (R²={r2:.3f})")
    opt_results.append((name, m.loss_curve_[-1], r2, m.n_iter_))

ax.set_xlabel("epoch"); ax.set_ylabel("training loss (MSE)")
ax.set_yscale("log"); ax.set_title("Optimiser convergence")
ax.legend(); ax.grid(True, alpha=0.3); plt.show()

pd.DataFrame(opt_results, columns=["optimiser", "final_train_loss", "test_R2", "epochs"])


## 3 · Batch size tradeoff

Gradient noise $\propto 1/\sqrt{B}$. Small batches → noisy but escape saddle points. Large batches → fast per epoch but sharper minima / worse generalisation.


In [ ]:
bs_list = [16, 64, 256, 1024]
bs_rows = []

for bs in bs_list:
    m = MLPRegressor(hidden_layer_sizes=(64, 32), solver="adam",
                     learning_rate_init=1e-3, batch_size=bs,
                     max_iter=50, random_state=42)
    t0 = time.time(); m.fit(X_tr, y_tr); dt = time.time() - t0
    r2 = r2_score(y_te, m.predict(X_te))
    bs_rows.append((bs, dt / m.n_iter_, r2, m.n_iter_))

df_bs = pd.DataFrame(bs_rows, columns=["batch_size", "sec_per_epoch", "test_R2", "epochs"])

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(df_bs.batch_size, df_bs.sec_per_epoch, "o-", color="#E67E22", label="sec / epoch")
ax1.set_xscale("log", base=2); ax1.set_xlabel("batch size (log₂)")
ax1.set_ylabel("seconds per epoch", color="#E67E22")
ax2 = ax1.twinx()
ax2.plot(df_bs.batch_size, df_bs.test_R2, "s-", color="#2E86C1", label="test R²")
ax2.set_ylabel("test R²", color="#2E86C1")
plt.title("Batch size: speed vs generalisation")
plt.tight_layout(); plt.show()

df_bs


## 4 · Initialiser — He vs small-random

Two tiny manual MLPs (8 → 32 → 1) trained with full-batch gradient descent on MSE.

- **He:** $W \sim \mathcal{N}(0, \sqrt{2/n_\text{in}})$ — preserves activation variance through ReLU layers.
- **Small random:** $W \sim \mathcal{N}(0, 0.01)$ — activations collapse to ~0; gradients vanish; training barely moves.


In [ ]:
def train_manual_mlp(init="he", steps=200, lr=1e-2, seed=0):
    rng = np.random.default_rng(seed)
    n_in, n_h = 8, 32
    if init == "he":
        W1 = rng.normal(0, np.sqrt(2 / n_in), (n_in, n_h))
        W2 = rng.normal(0, np.sqrt(2 / n_h),  (n_h, 1))
    else:
        W1 = rng.normal(0, 0.01, (n_in, n_h))
        W2 = rng.normal(0, 0.01, (n_h, 1))
    b1 = np.zeros(n_h); b2 = np.zeros(1)

    y = y_tr.reshape(-1, 1)
    losses = []
    for _ in range(steps):
        z1 = X_tr @ W1 + b1
        h1 = np.maximum(0, z1)
        yhat = h1 @ W2 + b2
        diff = yhat - y
        losses.append((diff ** 2).mean())
        # backward
        g_yhat = 2 * diff / len(y)
        gW2 = h1.T @ g_yhat;  gb2 = g_yhat.sum(0)
        g_h1 = g_yhat @ W2.T * (z1 > 0)
        gW1 = X_tr.T @ g_h1;  gb1 = g_h1.sum(0)
        # SGD update
        W1 -= lr * gW1; b1 -= lr * gb1
        W2 -= lr * gW2; b2 -= lr * gb2
    return losses

loss_he    = train_manual_mlp("he")
loss_small = train_manual_mlp("small")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(loss_he,    label="He init", color="#27AE60", lw=2)
ax.plot(loss_small, label="Small random  N(0, 0.01)", color="#C0392B", lw=2)
ax.set_xlabel("step"); ax.set_ylabel("MSE loss")
ax.set_title("Initialiser effect on training"); ax.set_yscale("log")
ax.legend(); ax.grid(True, alpha=0.3); plt.show()


## 5 · Regularisation sweep (L2 / weight decay)

`sklearn.neural_network.MLPRegressor` doesn't expose a true dropout switch, so we sweep its L2 penalty `alpha` instead — which plays the same role of penalising large weights (see Ch.6 for proper dropout in Keras/PyTorch). Expected: too small → overfits; too large → underfits.


In [ ]:
alphas = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
rows = []
for a in alphas:
    m = MLPRegressor(hidden_layer_sizes=(128, 64), solver="adam",
                     learning_rate_init=1e-3, alpha=a,
                     max_iter=120, random_state=42)
    m.fit(X_tr, y_tr)
    rows.append((a, r2_score(y_tr, m.predict(X_tr)),
                    r2_score(y_te, m.predict(X_te))))
df_a = pd.DataFrame(rows, columns=["alpha", "train_R2", "test_R2"])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_a.alpha, df_a.train_R2, "o-", label="train R²", color="#2E86C1")
ax.plot(df_a.alpha, df_a.test_R2,  "s-", label="test R²",  color="#C0392B")
ax.set_xscale("log"); ax.set_xlabel("L2 strength α (log)"); ax.set_ylabel("R²")
ax.set_title("Regularisation sweep — bias / variance tradeoff")
ax.legend(); ax.grid(True, alpha=0.3); plt.show()
df_a


## 6 · Loss functions visualised

You don't *tune* the loss — you *choose* it based on the target. These are the four most-used shapes you'll see in practice.


In [ ]:
r = np.linspace(-3, 3, 300)
mse   = r ** 2
mae   = np.abs(r)
huber = np.where(np.abs(r) < 1, 0.5 * r ** 2, np.abs(r) - 0.5)

p = np.linspace(0.01, 0.99, 300)
ce_pos = -np.log(p)          # loss when y=1
ce_neg = -np.log(1 - p)      # loss when y=0

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.5))

axL.plot(r, mse,   label="MSE",   lw=2)
axL.plot(r, mae,   label="MAE",   lw=2)
axL.plot(r, huber, label="Huber (δ=1)", lw=2)
axL.set_xlabel("residual y − ŷ"); axL.set_ylabel("loss")
axL.set_title("Regression losses"); axL.legend(); axL.grid(alpha=0.3)

axR.plot(p, ce_pos, label="y = 1: −log(p̂)", lw=2, color="#27AE60")
axR.plot(p, ce_neg, label="y = 0: −log(1−p̂)", lw=2, color="#C0392B")
axR.set_xlabel("predicted probability p̂"); axR.set_ylabel("loss")
axR.set_title("Binary cross-entropy"); axR.legend(); axR.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 7 · Depth sweep (number of layers)

Layer types carry the inductive bias (Dense, Conv, RNN, Attention). Given we're on tabular data → Dense. Here we vary **depth** (how many stacked Dense layers).


In [ ]:
depths = [1, 2, 3, 4, 6, 8]
depth_rows = []
for d in depths:
    m = MLPRegressor(hidden_layer_sizes=(64,) * d, solver="adam",
                     learning_rate_init=1e-3, max_iter=80, random_state=42)
    m.fit(X_tr, y_tr)
    depth_rows.append((d, r2_score(y_te, m.predict(X_te)), m.n_iter_))

df_d = pd.DataFrame(depth_rows, columns=["depth", "test_R2", "epochs"])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_d.depth, df_d.test_R2, "o-", lw=2, ms=8)
ax.set_xlabel("# hidden layers (each width 64)"); ax.set_ylabel("test R²")
ax.set_title("Depth vs generalisation — diminishing returns without skip connections")
ax.grid(alpha=0.3); plt.show()
df_d


## 8 · Width sweep (units per layer)

Width is often as important as depth. Pattern: `(W, W//2)` — widen to build representation, then compress to output.


In [ ]:
widths = [8, 16, 32, 64, 128, 256]
width_rows = []
for w in widths:
    m = MLPRegressor(hidden_layer_sizes=(w, max(w // 2, 4)), solver="adam",
                     learning_rate_init=1e-3, max_iter=80, random_state=42)
    m.fit(X_tr, y_tr)
    n_params = sum(c.size for c in m.coefs_) + sum(b.size for b in m.intercepts_)
    width_rows.append((w, r2_score(y_te, m.predict(X_te)), n_params))

df_w = pd.DataFrame(width_rows, columns=["width", "test_R2", "n_params"])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_w.width, df_w.test_R2, "o-", lw=2, ms=8, color="#8E44AD")
ax.set_xscale("log", base=2); ax.set_xlabel("hidden width (log₂)")
ax.set_ylabel("test R²"); ax.set_title("Width vs generalisation")
ax.grid(alpha=0.3); plt.show()
df_w


## 9 · Learning curves — when does more data help?

Plot train & validation R² against the number of training samples.

- **Train high, val low, gap still narrowing** → high variance → **more data helps**.
- **Train and val both plateau at a low score, small gap** → high bias → more data **won't** help; need a bigger model / better features.


In [ ]:
sizes = [500, 1000, 2000, 4000, 8000, 16000]
lc_rows = []
for n in sizes:
    n = min(n, len(X_tr))
    m = MLPRegressor(hidden_layer_sizes=(64, 32), solver="adam",
                     learning_rate_init=1e-3, max_iter=120, random_state=42)
    m.fit(X_tr[:n], y_tr[:n])
    lc_rows.append((n,
                    r2_score(y_tr[:n], m.predict(X_tr[:n])),
                    r2_score(y_te,      m.predict(X_te))))
df_lc = pd.DataFrame(lc_rows, columns=["n_samples", "train_R2", "test_R2"])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_lc.n_samples, df_lc.train_R2, "o-", label="train R²", color="#2E86C1")
ax.plot(df_lc.n_samples, df_lc.test_R2,  "s-", label="test R²",  color="#C0392B")
ax.fill_between(df_lc.n_samples, df_lc.test_R2, df_lc.train_R2, alpha=0.15, color="#C0392B")
ax.set_xlabel("# training samples"); ax.set_ylabel("R²")
ax.set_title("Learning curve — diagnose bias vs variance")
ax.legend(); ax.grid(alpha=0.3); plt.show()
df_lc


## Exercises

1. **LR-range test.** Starting at $\eta = 10^{-7}$, multiply $\eta$ by 1.1 after every mini-batch and log the training loss. Plot `loss` vs `log(η)`. The "sweet spot" is one order of magnitude below where the curve starts to rise. Implement this manually with a single-loop `SGDRegressor` or a custom training loop in NumPy.

2. **AdamW vs Adam+L2.** Rebuild this experiment using PyTorch (or `tf.keras.optimizers.AdamW`). Compare final test R² at identical `weight_decay` values. Expected: AdamW generalises a touch better on this tabular task.

3. **Manual early stopping.** Loop over epochs, track validation loss each epoch, stop once validation loss has not improved for `patience=5` epochs, and restore the best-seen weights. Compare the resulting test R² against `MLPRegressor(..., early_stopping=True)`.
